# Самостоятельная работа: языковая модель на Elman RNN

В этом ноутбуке требуется самостоятельно собрать словарную языковую
модель следующего слова. Учебный маршрут совпадает с подробным
примером, но все содержательные части реализации отмечены только
внутри кодовых ячеек «TODO(student)». Здесь используется отдельный
корпус русской классической литературы RuHeritage-Corpus, поэтому
данные, жанр и словарь не совпадают с примером.

## Маршрут самостоятельной работы

Выполняйте ячейки по порядку. В каждой спецификации указаны доступные
переменные, входы, ожидаемые формы и контрольная точка. Не вставляйте
готовое решение из примера: воспроизведите необходимые преобразования
самостоятельно. Индексы токенов являются входом модели, а целевые
индексы следующего токена — вторым аргументом функции потерь. Меток
классов документа в этой работе нет. Каждый этап имеет одну и ту же
структуру: цель, входы, действия, ожидаемый результат и контрольная
точка. Если проверка не пройдена, не переходите к следующему этапу.

## Этап 1. Настройка самостоятельного эксперимента

**Цель:** зафиксировать независимую от примера постановку. Константы
выбирают RuHeritage-Corpus, его каталог данных «data», закреплённую
ревизию и поле «text». Первые 1 200 документов этого *другого*
потока достаточны для учебного CPU-эксперимента.

**Результат этапа:** определены все размеры модели и специальные
индексы. **Проверьте:** PAD, UNK, BOS и EOS равны соответственно
0, 1, 2 и 3; «MAX_SEQUENCE_LENGTH» равна 32. Эти числа — часть
контракта следующих этапов: индекс 0 будет маскироваться в потере, а
размерности 128 и два рекуррентных слоя определяют формы параметров.

In [ ]:
from __future__ import annotations

import copy
import math
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
DATASET_NAME = 'maxzt/RuHeritage-Corpus'
DATASET_REVISION = 'b2d01eaad3ddb119fe10c4cb17a86e47415a77cb'
DATASET_DATA_DIR = 'data'
TEXT_FIELD = 'text'
START_DOCUMENT = 0
MAX_DOCUMENTS = 1_200
MAX_WORDS_PER_DOCUMENT = 160
MAX_ORDINARY_WORDS = 6_000
MAX_SEQUENCE_LENGTH = 32
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
BATCH_SIZE = 64
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 12
PATIENCE = 3

SPECIAL_TOKENS = ['<pad>', '<unk>', '<bos>', '<eos>']
PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX = range(4)
DEVICE = torch.device('cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Этап 2. Получение литературных документов и нормализация

**Цель:** превратить поток записей литературного корпуса в два списка
коротких последовательностей слов. **Входы:** «DATASET_NAME»,
«DATASET_REVISION», «DATASET_DATA_DIR», «TEXT_FIELD» и строка поля
«text». При загрузке передайте каталог данных «data», закреплённую
ревизию, «split='train'» и потоковый режим.

**Действие:** прочитайте только первые 1 200 документов, нормализуйте
текст, выделите русские словарные токены и оставьте не более 160 слов
на документ. Каждый шестой непустой документ направьте в валидацию.

Направляйте каждый шестой непустой документ в валидацию, остальные —
в обучение. Результат должен состоять из двух непустых списков
документов, где каждый документ является списком строк. Выведите
размеры и пример токенов. **Математический смысл:** в этот момент
строки превращаются в последовательности дискретных символов, но
ещё не в числа и не в векторы. **Контрольная точка:** не скачивается
весь корпус; обе части непусты; используется поле «text» именно
RuHeritage-Corpus. Если документ после нормализации слишком короток
для пары «вход — следующий токен», его разумно исключить до разбиения
на окна.

In [ ]:
# TODO(student): определите регулярное выражение и функцию нормализации.
# TODO(student): потоково получите train_documents и val_documents.
# Ожидаемый тип: list[list[str]] для обеих переменных.
raise NotImplementedError('Реализуйте потоковую загрузку и токенизацию.')

## Этап 3. Построение словаря и пар обучения

**Цель:** получить конечное индексное представление текста и
сформировать учебные примеры. **Доступные данные:** «train_documents», «val_documents»,
«SPECIAL_TOKENS», «MAX_ORDINARY_WORDS», «MAX_SEQUENCE_LENGTH».
**Требуется:** построить словарь только по обучающим документам с
резервированием индексов 0–3; незнакомые слова в обеих частях должны
переходить в «<unk>».

**Действие:** затем добавьте «<bos>» и «<eos>», разбейте длинный документ на окна не
длиннее 33 индексов и создайте пары «вход — цель» сдвигом на один
токен. Один объект Dataset должен вернуть два одномерных тензора
одинаковой длины, не превышающей 32. Контрольная точка: для пары
«[bos, a, b, eos]» вход равен «[bos, a, b]», а цель — «[a, b, eos]».
**Математический смысл:** второй массив — не метка документа, а
правильный класс следующего слова на каждой позиции. **Формы:** до
объединения в мини-пакет обе части одной пары одномерны, $(L)$, где
$1\leq L\leq32$; только функция объединения добавит ось $B$.

In [ ]:
# TODO(student): постройте stoi, itos и VOCAB_SIZE без утечки валидации.
# TODO(student): создайте train_pairs и val_pairs со сдвигом на один токен.
raise NotImplementedError('Реализуйте словарь и сдвинутые пары.')

## Этап 4. Мини-пакет и маска технического заполнения

**Цель:** сделать пары одинаковой формы внутри одного мини-пакета.
**Вход:** список пар переменной длины. **Требуется:** написать Dataset
и функцию «collate_fn», которые вернут «input_ids» и «target_ids»
формы $(B,T)$ с $T\leq32$. Короткие строки дополните индексом
«PAD_IDX=0» справа. Создайте обучающий и валидационный DataLoader;
у обучающего допустимо перемешивание, у валидационного — нет.

**Результат:** первая полученная пара тензоров имеет одинаковую
двумерную форму, а заполнение присутствует только справа. Не путайте
«target_ids» с метками классов: это правильные следующие слова.
**Контрольная точка:** число позиций, где «target_ids» не равно
«PAD_IDX», положительно; именно эти позиции позднее войдут в среднюю
кросс-энтропию.

In [ ]:
# TODO(student): реализуйте LanguageModelDataset и collate_language_batch.
# TODO(student): получите train_loader, val_loader и диагностический мини-пакет.
raise NotImplementedError('Реализуйте дополнение мини-пакета.')

## Этап 5. Ручная рекуррентная сеть и прогноз логитов

**Цель:** реализовать отображение индексов слов в оценки следующего
слова. **Вход модели:** «input_ids» формы $(B,T)$ со значениями от 0 до
$V-1$. **Требуемая архитектура:**
$(B,T)\rightarrow\operatorname{Embedding}(V,128)\rightarrow
\operatorname{ElmanRNN}(128,2)\rightarrow\operatorname{Linear}(128,V)$.

**Действие:** реализуйте собственный рекуррентный блок с обучаемыми входными и
рекуррентными весами, смещением и функцией tanh. Затем реализуйте два
последовательных слоя и выходную линейную проекцию. Не используйте
готовый «nn.RNN». Контрольная точка одного прямого прохода:
эмбеддинги и состояния имеют форму $(B,T,128)$, логиты — $(B,T,V)$.
Логиты ещё не являются вероятностями: это $V$ вещественных оценок,
к которым функция потерь внутренне применит softmax.

**Порядок вычислений:** индекс выбирает строку таблицы эмбеддингов;
на каждом времени рекуррентный блок объединяет этот вектор с прежним
состоянием; верхний слой получает результат нижнего; линейный слой
создаёт оценки всех слов. Результат модели обязан зависеть от
параметров класса, иначе «backward()» не сможет найти направление
их изменения.

In [ ]:
class ElmanRNNUnit(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        # TODO(student): создайте обучаемые параметры перехода.
        raise NotImplementedError('Реализуйте один рекуррентный переход.')

    def forward(self, x_t: torch.Tensor, h_previous: torch.Tensor) -> torch.Tensor:
        # TODO(student): верните новое скрытое состояние формы (B, 128).
        raise NotImplementedError('Реализуйте прямой проход ElmanRNNUnit.')


class ElmanRNN(nn.Module):
    def __init__(self, hidden_size: int, num_layers: int):
        super().__init__()
        # TODO(student): создайте два рекуррентных слоя.

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        # TODO(student): разверните вычисление по времени и слоям.
        raise NotImplementedError('Реализуйте последовательный рекуррентный проход.')


class ElmanLanguageModel(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        # TODO(student): Embedding с padding_idx=0, ElmanRNN и Linear.

    def forward(self, input_ids: torch.Tensor):
        # TODO(student): верните логиты формы (B, T, V).
        raise NotImplementedError('Реализуйте языковую модель.')

## Этап 6. Обучение по сдвинутым целям и ранняя остановка

**Цель:** изменить параметры так, чтобы правильный следующий токен
получал больший логит. **Входы каждого шага:** «input_ids» и «target_ids» формы $(B,T)$.
**Требуется:** получить логиты $(B,T,V)$, развернуть логиты и цели по
позициям, создать «CrossEntropyLoss(ignore_index=PAD_IDX)» и AdamW.
Усредняйте потерю только по незаполненным позициям; сохраняйте лучшее
состояние по валидационной потере; завершайте обучение после трёх эпох
без улучшения, но не позже 12-й.

**Порядок вычислений:** в обучающем цикле очистите градиенты, выполните прямой проход,
«backward()» и «step()». На валидации обновление параметров запрещено.
Храните списки «history['loss']» и «history['val_loss']». Контрольная
точка: все значения конечны, а потеря обучения в целом снижается.
При вычислении итоговой эпохи суммируйте отрицательные логарифмы и
число реальных токенов по всем мини-пакетам отдельно: среднее средних
при разных длинах мини-пакетов дало бы смещённую оценку.

In [ ]:
# TODO(student): создайте model, criterion и optimizer.
# TODO(student): реализуйте обучение, валидацию, history и early stopping.
# TODO(student): восстановите сохранённое лучшее состояние.
raise NotImplementedError('Реализуйте обучение с маскированием PAD.')

## Этап 7. Оценка, перплексия и осмысленная интерпретация

**Цель:** проверить модель на данных, по которым не происходило
обновление параметров. **Вход:** лучшая обученная модель и валидационный DataLoader.
**Требуется:** вычислить среднюю кросс-энтропию по токенам и
перплексию $\exp(\mathcal L)$, построить графики «loss» и «val_loss»,
получить top-$k$ продолжений короткого русского запроса и выполнить
жадную генерацию до EOS или лимита.

Контрольная точка: перплексия — конечное положительное число; top-$k$
содержит слова из «itos»; цикл генерации имеет явный предел шагов.
В отчёте объясните, почему маленькая модель может выдавать частые, но
не всегда содержательные продолжения. Сопоставьте каждый top-$k$
прогноз с тем, что логиты последней позиции сначала превращаются в
распределение softmax, а затем сортируются по вероятности.

In [ ]:
# TODO(student): реализуйте оценку, график, top-k прогноз и генерацию.
raise NotImplementedError('Реализуйте оценку языковой модели.')

## Этап 8. Самопроверка перед сдачей

- Использован RuHeritage-Corpus, его поле «text», каталог данных
  «data» и закреплённая ревизия.
- Словарь построен только по обучающей части; индексы PAD, UNK, BOS,
  EOS равны 0, 1, 2, 3.
- «input_ids» и «target_ids» имеют форму $(B,T)$, $T\leq32$.
- Эмбеддинги и скрытые состояния имеют форму $(B,T,128)$; логиты —
  $(B,T,V)$.
- PAD исключён из кросс-энтропии, а не заменён на обычную цель.
- Потери и перплексия конечны; сохранена лучшая валидационная модель.
- В коде нет TensorFlow, Keras и готового «nn.RNN».